In [2]:
%matplotlib inline
import matplotlib.pyplot as plt

import drjit as dr
import mitsuba as mi

# Import or install Sionna
try:
    import sionna.rt
except ImportError as e:
    import os
    os.system("pip install sionna-rt")
    import sionna.rt

no_preview = True # Toggle to False to use the preview widget
                  # instead of rendering for scene visualization

from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, Camera,\
                      PathSolver, ITURadioMaterial, SceneObject

In [3]:
#pip install jupyter

In [4]:
scene= load_scene("/home/xzhang/sionna_XJ/528factory/528newfactory.xml",merge_shapes=False)

In [5]:
scene.frequency = 5.8e9 

In [6]:
scene.preview()

Renderer(camera=PerspectiveCamera(aspect=1.31, children=(DirectionalLight(intensity=0.25, position=(0.0, 0.0, …

In [7]:
import numpy as np
#add transmitter
scene.remove("tx")
scene.remove("rx")
#TX= scene.add(Transmitter("tx",position=[0,0,20], display_radius=3, orientation=[0, np.pi/6, np.pi/6]))
TX= scene.add(Transmitter("tx",position=[5,-23,18], display_radius=3, orientation=[0, 0, 0]))
rx=scene.add(Receiver("rx", position=[5, -23, 18], display_radius=3,orientation=[0, 0, 0]))
#rx=scene.add(Receiver("rx", position=[-10, -11, 1.2], display_radius=3,orientation=[0, -np.pi/2, 0]))
# tx.look_at(rx)
# Set the transmit and receive antenna arrays
scene.tx_array = PlanarArray(num_cols=1,
                             num_rows=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern="iso",
                             polarization="VH")
scene.rx_array = scene.tx_array

In [8]:
from sionna.rt import subcarrier_frequencies
num_ofdm_symbols = 140
num_subcarriers = 4096
subcarrier_spacing = 30e3
frequencies = subcarrier_frequencies(num_subcarriers, subcarrier_spacing)
ofdm_symbol_duration = 1/subcarrier_spacing


In [9]:
#add material
car_material = ITURadioMaterial("car-material",
                                "metal",
                                thickness=0.01,
                                color=(0.1, 0.2, 0.3),
                                   #scattering_coefficient=0.1)
                                scattering_coefficient=0.1)

human_material = ITURadioMaterial("human-material",
                                "metal",
                                #thickness=0.01,
                                thickness=0.05,
                                color=(0.1, 0.4, 0),
                                   #scattering_coefficient=0.2)
                                scattering_coefficient=0.4)

roof_material = ITURadioMaterial("roof",
                                "ceiling_board",
                                thickness=0.01,
                                color=(0.1, 0.4, 0),
                                scattering_coefficient=0.01)

wall_material = ITURadioMaterial("wall",
                                "metal",
                                thickness=0.01,
                                color=(0.1, 0.4, 0),
                                scattering_coefficient=0.01)
  # Assuming "Pedestrian" is the object's name

human = scene.get("Low_poly_man_wearing_suit")
human.radio_material = human_material
human.scaling = 1.0
human.velocity= [0,0,0]
AGV_1 = scene.get("Cube_661")
AGV_1.radio_material = car_material
AGV_1.scaling = 1.0
AGV_1.velocity= [200,0,0]


Roof1 = scene.get("Cube_605")
Roof1.radio_material = roof_material
Roof2 = scene.get("Cube_606")
Roof2.radio_material = roof_material

wall1 =scene.get("Cube_001")
wall1.radio_material = wall_material
wall2 =scene.get("Cube_002")
wall2.radio_material = wall_material
wall3 =scene.get("Cube_003")
wall3.radio_material = wall_material
wall4 =scene.get("Cube_004")
wall4.radio_material = wall_material
# print(f"Pedestrian updated material: {pedestrian.radio_material.name}")

In [10]:
delay_resolution = ofdm_symbol_duration/num_subcarriers
doppler_resolution = subcarrier_spacing/num_ofdm_symbols
print("Delay   resolution (ns): ", int(delay_resolution/1e-9))
print("Doppler resolution (Hz): ", int(doppler_resolution))

Delay   resolution (ns):  8
Doppler resolution (Hz):  214


In [11]:
from sionna.rt.utils import r_hat, subcarrier_frequencies
max_depth=2
AGV_1.position=[-10, -11.191, 1.288]
p_solver = PathSolver()
paths = p_solver(scene, max_depth=1,max_num_paths_per_src=10000, diffuse_reflection=True, specular_reflection=True, refraction=True, synthetic_array=False)
#paths = p_solver(scene, max_depth=2, max_num_paths_per_src=1000,diffuse_reflection=True, specular_reflection=True, refraction=True, synthetic_array=False)
scene.preview(paths=paths,clip_at =10)

Renderer(camera=PerspectiveCamera(aspect=1.31, children=(DirectionalLight(intensity=0.25, matrixWorldNeedsUpda…

In [12]:
print(AGV_1.position)
p_solver = PathSolver()

[[-10, -11.191, 1.288]]


In [13]:
tx = scene.get("tx")
#new_pos = tx.position + np.array([0, 0, -5])
cam_pos = [25, 25, 18]
#look_pos = [0, 0, 3]
zoom_cam = Camera(position=cam_pos, look_at=AGV_1.position)
#scene.render(camera=zoom_cam,paths=paths,clip_at =18);

In [14]:

    # Compute channel frequency response with time evolution
frequencies = subcarrier_frequencies(num_subcarriers, subcarrier_spacing)
    
h = paths.cfr(frequencies=frequencies,
                  sampling_frequency=1/ofdm_symbol_duration,
                  num_time_steps=num_ofdm_symbols,
                  normalize_delays=False, normalize=False, out_type="numpy")
    
    

a, tau = paths.cir(normalize_delays=False, out_type="numpy")
t = tau[0, 0, :] / 1e-9  # Scale to ns
a_abs = np.abs(a)[0, 0, 0, 0, :, 0]
#a_abs = np.nan_to_num(a_abs,nan=0.0)
a_max = np.max(a_abs)
print(a_abs)

[2.92307705e-05 8.18291563e-04 4.14530717e-07            nan
            nan 7.68541639e-08            nan 1.00475314e-04
 6.88125074e-05 4.24001519e-05 2.30227968e-07 2.06276084e-07
 1.09309561e-07            nan 3.39822293e-07 3.40019369e-07
 3.09041411e-07            nan 2.23188090e-06 3.52478622e-07
 2.04707774e-07 2.23374911e-07 3.86000664e-07 3.51421960e-07
            nan 1.32304265e-07 1.96727740e-07 1.84925284e-07
 2.38012035e-07 8.16470518e-08 2.04011059e-07 2.48862705e-07
 2.04063070e-07 2.41728856e-07 2.40784402e-07 2.50737600e-07
 2.24914203e-07 2.70034718e-07 1.67473146e-07 2.25061584e-07
 2.18899754e-06 2.18556224e-07 2.43961239e-07 2.52278483e-07
 2.06290835e-07 2.33476698e-07            nan 2.42535151e-07]
